<a href="https://colab.research.google.com/github/InduwaraGayashan001/LangChain/blob/main/Agents_%26_Tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Setup

In [2]:
from google.colab import userdata

In [ ]:
!pip install langchain_huggingface langchain-community langchain-openai

# Without Agents

In [26]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=userdata.get('GITHUB_TOKEN'),
    base_url="https://models.github.ai/inference"
)
llm.invoke("What is the time now in Paris? (I'm in Sri Lanaka)").content

"To find the current time in Paris, France, you can calculate the time difference between Sri Lanka and Paris. Sri Lanka operates on Sri Lanka Standard Time (SLST), which is UTC+5:30. Paris operates on Central European Time (CET), which is UTC+1, and during Daylight Saving Time (from the last Sunday in March to the last Sunday in October), it uses Central European Summer Time (CEST), which is UTC+2.\n\nAs of now, if it is during Daylight Saving Time (October 2023), Paris would be UTC+2. If not, it would be UTC+1.\n\n1. If it's currently Daylight Saving Time in Paris (UTC+2):\n   - Time in Paris = Time in Sri Lanka - 3 hours and 30 minutes.\n\n2. If it's not Daylight Saving Time (UTC+1):\n   - Time in Paris = Time in Sri Lanka - 4 hours and 30 minutes.\n\nPlease check the current local time in Sri Lanka and apply the appropriate time difference to find the time in Paris."

# Prompt Initialization

In [ ]:
!pip install langchainhub

In [27]:
from langchain_classic import hub

prompt = hub.pull("hwchase17/react")

# **Tool**

In [24]:
from langchain_classic.agents import tool
import datetime

@tool
def get_system_date_time(format: str = "%Y-%m-%d %H:%M:%S") -> str:
  """Return the current date and time in given format"""
  tz = datetime.timezone(datetime.timedelta(hours=5, minutes=30))
  current_date_time = datetime.datetime.now(tz)
  return current_date_time.strftime(format)

tools = [get_system_date_time]


# Agent

In [59]:
from langchain_classic.agents import create_react_agent, AgentExecutor

agent = create_react_agent(llm, tools, prompt)

agent_executor= AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({'input':"what is the time now in Paris?(I'm in Sri Lanka)"})



> Entering new AgentExecutor chain...
To answer the question, I need to determine the current time in Paris, France. Since I'm currently in Sri Lanka, I should first find out the current time in Sri Lanka and then convert it to the time in Paris, taking into account the time zone difference.

Action: get_system_date_time
Action Input: '%Y-%m-%d %H:%M:%S''2025-12-24 16:12:57'To find the current time in Paris, I need to determine the time zone of both Sri Lanka and Paris and then apply the correct time zone offset. 

Sri Lanka operates on Sri Lanka Standard Time (SLST), which is UTC+5:30. Paris operates on Central European Time (CET), which is UTC+1 during standard time and UTC+2 during daylight saving time (which usually runs from the last Sunday in March to the last Sunday in October).

Since today's date is December 24, Paris would be on standard time (UTC+1). Therefore, to convert the current time from Sri Lanka to Paris time, I need to subtract the difference in hours: 

1. Time d

{'input': "what is the time now in Paris?(I'm in Sri Lanka)",
 'output': 'The current time in Paris is 11:43 AM on December 24, 2025.'}

# Working with Multiple Tools

In [ ]:
!pip install wikipedia

In [60]:
import wikipedia

@tool
def search_wikipedia(query: str) -> str:
  """Search Wikipedia for given query"""
  try:
    return wikipedia.summary(query, sentences=2)
  except:
    return "No good Wikipedia Search Result was found"


new_tools = [search_wikipedia, get_system_date_time]

In [62]:
new_agent = create_react_agent(llm, new_tools, prompt)

new_agent_executor= AgentExecutor(agent=new_agent, tools=new_tools, verbose=True)

new_agent_executor.invoke({'input':"How many days left for the ICC T20 World Cup 2026?"})




> Entering new AgentExecutor chain...
To determine how many days are left for the ICC T20 World Cup 2026, I need to find the start date of the event. I will search for the ICC T20 World Cup 2026 on Wikipedia to get this information.  
Action: search_wikipedia  
Action Input: "ICC T20 World Cup 2026"  The 2026 ICC Men's T20 World Cup will be the tenth edition of the ICC Men's T20 World Cup, co-hosted by Board of Control for Cricket in India and Sri Lanka Cricket from 7 February to 8 March 2026. Sri Lanka had previously hosted the competition in 2012 and India in 2016.I now know the start date of the ICC T20 World Cup 2026, which is on February 7, 2026. Now, I need to calculate how many days are left until that date from today. 

Action: get_system_date_time  
Action Input: '%Y-%m-%d'  '2025-12-24'Today is December 24, 2025. The ICC T20 World Cup 2026 starts on February 7, 2026. To find out how many days are left, I will calculate the difference between these two dates.

1. The date to

{'input': 'How many days left for the ICC T20 World Cup 2026?',
 'output': 'There are 45 days left until the ICC T20 World Cup 2026.'}